# SegmentAE Example: Grid Search Optimization

This notebook demonstrates how to use SegmentAE_Optimizer for automated grid search to find the optimal configuration across multiple autoencoders, clustering methods, and threshold ratios.

## Features demonstrated:
- EnsembleAutoencoder with multiple architectures
- Custom basic autoencoder implementation
- Automated grid search optimization
- Performance leaderboard and best configuration selection

## Import Dependencies

In [ ]:
import warnings

import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

from segmentae.autoencoders import EnsembleAutoencoder
from segmentae.data_sources import load_dataset
from segmentae.metrics import metrics_classification
from segmentae.optimization import SegmentAE_Optimizer
from segmentae.processing.preprocessing import Preprocessing

warnings.filterwarnings("ignore")

## Data Loading

In [ ]:
train, test, target = load_dataset(
    dataset_selection="htru2_dataset",  # Options: 'default_credit_card', 'htru2_dataset', 'shuttle_148'
    split_ratio=0.75,
)

X_train = train.drop(columns=[target]).copy()
X_test = test.drop(columns=[target]).copy()
y_test = test[target].astype(int)

print(f"Dataset: HTRU2 Pulsar Detection")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Target classes: {sorted(y_test.unique())}")

## Data Preprocessing

In [ ]:
pr = Preprocessing(
    encoder="IFrequencyEncoder",  # "LabelEncoder", "OneHotEncoder"
    scaler="StandardScaler",  # "MinMaxScaler", "RobustScaler"
    imputer=None,
)

pr.fit(X=X_train)
X_train = pr.transform(X=X_train)
X_test = pr.transform(X=X_test)

print("✓ Preprocessing completed")
print(f"  Processed shape: {X_train.shape}")

## Check GPU Availability

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices("GPU")))
if len(tf.config.experimental.list_physical_devices("GPU")) > 0:
    print("✓ GPU acceleration available")
else:
    print("⚠️ Running on CPU only")

## Ensemble Autoencoder Architecture

Build an ensemble autoencoder that combines multiple architectures for robustness.

In [ ]:
# EnsembleAutoencoder combines multiple architectures for robustness
print("Building Ensemble Autoencoder...")
ensembleAutoencoder = EnsembleAutoencoder(
    n_autoencoders=3,
    hidden_dims=[[24, 12, 8, 4], [64, 32, 16, 4], [20, 10, 5]],
    encoder_activations=["relu", "tanh", "relu"],
    decoder_activations=["relu", "tanh", "relu"],
    optimizers=["adam", "sgd", "rmsprop"],
    learning_rates=[0.001, 0.01, 0.005],
    epochs_list=[10, 15, 20],
    val_size_list=[0.15, 0.15, 0.15],
    stopping_patients=[30, 30, 15],
    dropout_rates=[0.1, 0.2, 0.3],
    batch_sizes=[None, None, 128],
    use_batch_norm=[False, False, False],
)

print("  Configuration:")
print(f"    - Number of autoencoders: {ensembleAutoencoder.n_autoencoders}")
print(f"    - Architectures: {ensembleAutoencoder.hidden_dims}")
print(f"    - Optimizers: {ensembleAutoencoder.optimizers}")

In [ ]:
# Train the ensemble
print("Training ensemble autoencoder...")
ensembleAutoencoder.fit(X_train)
print("✓ Ensemble autoencoder trained successfully!")

## Custom Basic Autoencoder

Create a simple custom autoencoder for comparison.

In [ ]:
print("Building Custom Basic Autoencoder...")

# Define encoder architecture
encoder = Sequential(
    [
        Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
        Dense(24, activation="relu"),
        Dense(16, activation="relu"),
        Dense(8, activation="relu"),
    ]
)

# Define decoder architecture
decoder = Sequential(
    [
        Dense(16, activation="relu", input_shape=(8,)),
        Dense(24, activation="relu"),
        Dense(32, activation="relu"),
        Dense(X_train.shape[1], activation="sigmoid"),
    ]
)

# Combine into autoencoder
basicAutoencoder = Sequential([encoder, decoder])
basicAutoencoder.compile(optimizer="adam", loss="mse")

print(f"  Architecture: Input → 32 → 24 → 16 → 8 → 16 → 24 → 32 → Output")
print(f"  Total parameters: {basicAutoencoder.count_params():,}")

In [ ]:
# Train the model
print("Training basic autoencoder...")
basic_history = basicAutoencoder.fit(X_train, X_train, epochs=50, batch_size=None, validation_split=0.1, verbose=1)
print("✓ Basic autoencoder trained successfully!")

## Grid Search Optimization Setup

Configure the SegmentAE_Optimizer to test multiple configurations automatically.

In [ ]:
# Using string-based configuration (backward compatible)
print("Setting up SegmentAE Optimizer...")

optimizer = SegmentAE_Optimizer(
    autoencoder_models=[ensembleAutoencoder, basicAutoencoder],
    n_clusters_list=[1, 2, 3],
    cluster_models=["KMeans", "MiniBatchKMeans", "GMM"],
    threshold_ratios=[1, 1.5, 2, 4],
    performance_metric="Accuracy",
)

print("  Optimization configuration:")
print(f"    - Autoencoders: 2 models (Ensemble + Basic)")
print(f"    - Cluster numbers: {optimizer.n_clusters_list}")
print(f"    - Clustering methods: {optimizer.cluster_models}")
print(f"    - Threshold ratios: {optimizer.threshold_ratios}")
print(f"    - Optimization metric: {optimizer.performance_metric}")

total_configs = len(optimizer.autoencoder_models) * len(optimizer.n_clusters_list) * len(optimizer.cluster_models) * len(optimizer.threshold_ratios)
print(f"    - Total configurations to test: {total_configs}")

## Run Grid Search Optimization

Execute the automated optimization process.

In [ ]:
# Run optimization
print("\n Running grid search optimization...")
print("This may take a few minutes...\n")

sg = optimizer.optimize(X_train, X_test, y_test)

print("\n✓ Optimization completed!")

## Optimization Results Analysis

In [ ]:
# Access optimization results
preds_test = sg.preds_test
recon_metrics_test = sg.reconstruction_test
leaderboard = optimizer.leaderboard

print("=" * 60)
print("🏆 OPTIMIZATION RESULTS")
print("=" * 60)
print(f"\nBest Performance: {optimizer.best_performance:.4f}")
print(f"\nBest Configuration:")
print(f"  - Model: {'Ensemble' if optimizer.best_model == ensembleAutoencoder else 'Basic Custom'}")
print(f"  - Clusters: {optimizer.best_n_clusters}")
print(f"  - Clustering Algorithm: {optimizer.best_cluster_model}")
print(f"  - Threshold Ratio: {optimizer.best_threshold_ratio}")

## Performance Leaderboard

View the top performing configurations.

In [ ]:
print("\nTop 10 Configurations:")
print("=" * 80)

# Display leaderboard with better formatting
top_10 = leaderboard.head(10).copy()
top_10['Autoencoder_Type'] = top_10['Autoencoder'].apply(
    lambda x: 'Ensemble' if x == ensembleAutoencoder else 'Basic'
)

display_cols = ['Autoencoder_Type', 'Cluster Model', 'n_clusters', 'Threshold Ratio', 'Accuracy']
print(top_10[display_cols].to_string(index=False))

## Final Anomaly Detection with Optimal Configuration

In [ ]:
threshold = optimizer.best_threshold_ratio

predictions = sg.detections(input_data=X_test, threshold_ratio=threshold)

# Calculate final metrics
final_metrics = metrics_classification(y_test, predictions["Predicted Anomalies"])

print("\n" + "=" * 50)
print(" FINAL TEST PERFORMANCE")
print("=" * 50)
print(f"\nUsing optimal configuration:")
print(f"  - Threshold ratio: {threshold}")
print(f"  - Clustering: {optimizer.best_cluster_model} (n={optimizer.best_n_clusters})")
print(f"\nPerformance metrics:")
for metric, value in final_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Detection summary
n_anomalies = predictions["Predicted Anomalies"].sum()
anomaly_rate = n_anomalies / len(predictions)
print(f"\nDetection summary:")
print(f"  Anomalies detected: {n_anomalies:,} / {len(predictions):,}")
print(f"  Anomaly rate: {anomaly_rate:.2%}")

## Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Performance comparison by autoencoder type
plt.figure(figsize=(12, 5))

# Add autoencoder type to leaderboard for visualization
viz_data = leaderboard.copy()
viz_data['Autoencoder_Type'] = viz_data['Autoencoder'].apply(
    lambda x: 'Ensemble' if x == ensembleAutoencoder else 'Basic'
)

plt.subplot(1, 2, 1)
sns.boxplot(data=viz_data, x='Autoencoder_Type', y='Accuracy')
plt.title('Performance by Autoencoder Type')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
sns.boxplot(data=viz_data, x='Cluster Model', y='Accuracy')
plt.title('Performance by Clustering Method')
plt.xticks(rotation=45)
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()

print("\nVisualization shows performance distribution across different configurations")